In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
data = yf.download('SPY', progress=False, auto_adjust=False, start='1990-01-01', multi_level_index=False).reset_index()
data['Date'] = pd.to_datetime(data['Date'])
data.sort_values('Date', inplace=True)

data['r_m'] = data['Close'].pct_change(1)

# returns & momentum & reversal
data['chg_1d'] = np.log(data['Close'] / data['Close'].shift(1))
data['chg_2d'] = np.log(data['Close'] / data['Close'].shift(2))
data['chg_3d'] = np.log(data['Close'] / data['Close'].shift(3))

data['chg_1w'] = np.log(data['Close'] / data['Close'].shift(5))
data['chg_2w'] = np.log(data['Close'] / data['Close'].shift(10))

data['chg_1m'] = np.log(data['Close'] / data['Close'].shift(21))
data['chg_3m'] = np.log(data['Close'] / data['Close'].shift(21*3))
data['chg_6m'] = np.log(data['Close'] / data['Close'].shift(21*6))
data['chg_12m'] = np.log(data['Close'] / data['Close'].shift(21*12))

# volatility feats
data['volatility_1m'] = data['chg_1d'].rolling(21).std()
data['volatility_3m'] = data['chg_1d'].rolling(21*3).std()

data['volatility_of_volatility_1m'] = data['volatility_1m'].rolling(21).std()

data['neg_chg_1d'] = np.where(data['chg_1d'] < 0, data['chg_1d'], 0)
data['neg_volatility_1m'] = data['neg_chg_1d'].rolling(21).std()

# skew & kurt feats
data['skew_1m'] = data['chg_1d'].rolling(21).skew()
data['skew_3m'] = data['chg_1d'].rolling(21*3).skew()
data['kurt_1m'] = data['chg_1d'].rolling(21).kurt()
data['kurt_3m'] = data['chg_1d'].rolling(21*3).kurt()

# ma ratio
data['ma_ratio_price_1m'] = data['Close'] / data['Close'].rolling(21).mean()
data['ma_ratio_price_3m'] = data['Close'] / data['Close'].rolling(21*3).mean()

data['ma_ratio_volume_1m'] = data['Volume'] / data['Volume'].rolling(21).mean()
data['ma_ratio_volume_3m'] = data['Volume'] / data['Volume'].rolling(21*3).mean()

# Volume Pressure
data['volume_pressure_1m'] = data['Volume'] / data['ma_ratio_volume_1m']
data['volume_pressure_3m'] = data['Volume'] / data['ma_ratio_volume_3m']

# max loss
data['max_loss_1m'] = data['chg_1d'].rolling(21).min()
data['max_loss_3m'] = data['chg_1d'].rolling(21*3).min()

# drawdown & drawup
data['cumret'] = (data['chg_1d'] + 1).cumprod()
data['running_max_12m'] = data['cumret'].rolling(252).max()
data['drawdown_12m'] = data['cumret'] / data['running_max_12m'] - 1

data['running_min_12m'] = data['cumret'].rolling(252).min()
data['drawup_12m'] = data['cumret'] / data['running_min_12m'] - 1

# z score
data['price_mean'] = data['Close'].rolling(21).mean()
data['price_std'] = data['Close'].rolling(21).std()
data['volume_mean'] = data['Volume'].rolling(21).mean()
data['volume_std'] = data['Volume'].rolling(21).std()

data['z_price_1m'] = (data['Close'] - data['price_mean']) / data['price_std']
data['z_volume_1m'] = (data['Volume'] - data['volume_mean']) / data['volume_std']

# Trend exhaustion proxy
data['trend_slope_1m'] = (data['Close'] - data['Close'].shift(21)) / 21
data['trend_slope_3m'] = (data['Close'] - data['Close'].shift(63)) / 63

data.dropna(inplace=True)

del (
    data['Adj Close'], 
    data['neg_chg_1d'], 
    data['cumret'], 
    data['running_max_12m'], 
    data['running_min_12m'], 
    data['price_mean'], 
    data['price_std'], 
    data['volume_mean'], 
    data['volume_std']
)

data

,Date,Close,High,Low,Open,Volume,r_m,chg_1d,chg_2d,chg_3d,...,volume_pressure_1m,volume_pressure_3m,max_loss_1m,max_loss_3m,drawdown_12m,drawup_12m,z_price_1m,z_volume_1m,trend_slope_1m,trend_slope_3m
252,1994-01-27,47.750000,47.812500,47.343750,47.406250,344500,0.009247,0.009205,0.011850,0.011850,...,3.190286e+05,2.813746e+05,-0.005351,-0.012730,0.000000,0.097331,1.595221,0.104860,0.034226,0.014385
253,1994-01-28,47.875000,48.031250,47.875000,47.937500,356500,0.002618,0.002614,0.011819,0.014464,...,3.255333e+05,2.857524e+05,-0.005351,-0.012730,0.000000,0.100200,1.700551,0.127988,0.049107,0.016369
254,1994-01-31,48.218750,48.312500,48.000000,48.062500,313800,0.007180,0.007155,0.009769,0.018973,...,3.255762e+05,2.901556e+05,-0.003966,-0.012730,0.000000,0.108071,2.224505,-0.048673,0.077381,0.019841
255,1994-02-01,47.968750,48.156250,47.906250,48.156250,303600,-0.005185,-0.005198,0.001956,0.004571,...,2.942762e+05,2.908143e+05,-0.005198,-0.012730,-0.005198,0.102311,1.526933,0.048243,0.071429,0.016369
256,1994-02-02,48.281250,48.281250,48.093750,48.125000,307600,0.006515,0.006494,0.001295,0.008450,...,3.011000e+05,2.880921e+05,-0.005198,-0.010848,0.000000,0.109469,2.065743,0.034038,0.077381,0.030754
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8371,2026-05-04,718.010010,722.119995,714.989990,720.070007,51950600,-0.003663,-0.003670,-0.000905,0.008996,...,5.358751e+07,8.039438e+07,-0.006568,-0.018020,-0.003670,0.275056,0.982268,-0.116619,2.960952,0.358731
8372,2026-05-05,723.770020,725.039978,721.489990,721.770020,36933200,0.008022,0.007990,0.004320,0.007085,...,5.348405e+07,7.926785e+07,-0.006568,-0.018020,0.000000,0.285244,1.236473,-1.169182,3.087620,0.543492
8373,2026-05-06,733.830017,734.590027,727.820007,728.159973,53288900,0.013899,0.013804,0.021794,0.018124,...,5.268922e+07,7.844379e+07,-0.006568,-0.018020,0.000000,0.302986,1.801298,0.043956,3.552859,0.756191
8374,2026-05-07,731.580017,736.130005,729.750000,735.049988,51724600,-0.003066,-0.003071,0.010733,0.018723,...,5.069486e+07,7.746147e+07,-0.006568,-0.018020,-0.003071,0.293556,1.550265,0.103869,2.646191,0.856508


In [3]:
data.to_csv('data.csv', index=False)